# 18.10 蒸餾是否幫到學生？


蒸餾學生答對更多題，是否一定是教師訊號的功勞？如果它比基準多讀了一倍資料，或學生架構也變大，答案還不能歸因蒸餾。比較先固定[學生架構](https://birdhackor.github.io/tiny-perceptron-vlm/18.2.html)，把同一初始化、訓練資料與已讀token預算分成兩條路：普通SFT與加入蒸餾，最後在同一份獨立真值上評估。SFT是監督式微調，以理想助手回答教學生；加入蒸餾則另外讀教師的候選比例。這份獨立評估題沒有用來更新學生，答案真值也不由教師的猜測代填。

教師計算並不免費。假設兩種方法學生部分各花10秒，訓練100步、讀1000個token；蒸餾另花40秒教師forward（前向計算：把輸入交給教師，算出候選分數與比例），總時間就是50秒，不是10秒。按學生更新預算比較能回答「多給教師訊號有沒有幫忙」；按同樣總時間比較則回答另一個問題，蒸餾可能只能做較少更新。兩種公平應分開報。

下面的人工表沿用這個時間情境，兩個學生都設為6104個參數、100次更新、1000個訓練token。我們再假設四題中分別答對三題與四題，練習同時讀品質與成本；這些構造數字不是模型實測。

| 方法 | 答對數／四題 | 學生時間 | 教師時間 | 合計 |
| --- | ---: | ---: | ---: | ---: |
| SFT | 3/4（75%） | 10秒 | 0秒 | 10秒 |
| 蒸餾 | 4/4（100%） | 10秒 | 40秒 | 50秒 |

假設的蒸餾版答對更多，也多花教師時間。這張表只能說明報告應呈現什麼；要證實改善，必須保存真實訓練結果，再用足夠獨立題目與重複實驗檢查。重算正確率與總秒數的原Python放在本節最後。

評價不能只用「與教師回答一致率」。教師若給錯答案，學生模仿得更像可能反而退步；應用真值判內容，用格式規格判結構，並重跑既有誠實與安全情境。同樣正確率下，也可能一組善於短題、另一組在長題失敗，所以應按題型或輸入條件拆開看。

若教師先生成離線文字，成本改成資料生成與驗證，蒸餾訓練時不一定每步教師forward；仍要另外列出這些一次性成本。正式白盒訓練需要可載入、已檢查目標任務能力的教師權重；能力不足可以作失敗對照，但要明說，不能把本章的隨機教師檢查當作可靠教師。需要保存版本與[詞表對齊](https://birdhackor.github.io/tiny-perceptron-vlm/18.6.html)條件，讓比較能重現。

一輪屬性題的實際對照提供了另一個讀表例子：寬32的一層學生，CE與CE+KL使用同初始化、45道訓練題，各更新400次、44,985個助手回答及EOS目標。共同最後十題的完整答案都答對4/10，但真值回答NLL分別0.4393與0.5037；NLL是正確token機率的負自然log，沿共同69個回答／EOS目標平均，越低越好。教師十題答對5/10。這次沒有觀察到蒸餾提高答對數，亦不能由這個單一種子小任务推出蒸餾普遍無效。原始各寬度結果、成本與域外診斷留在選讀；域外題只說明這個教師的適用範圍，不用來判定整種方法。

練習把人工表中蒸餾的教師時間改成0。先算正確率是否改變、總時間剩多少，再用末尾Python把`teacher_seconds`改成0核對。結果是正確率不變、總時間變10秒；你只改了記帳假設，沒有訓練新學生。如果真的預先快取教師訊號，還應補報生成快取所花時間與儲存。

<details>
<summary>選讀：完整屬性對照、成本與域外診斷</summary>

正式屬性實驗把同一個寬度的初始化分為三支，各讀相同45道訓練題、batch16，完成400次更新與44,985個回答／EOS目標。寬16與寬32分開比較，全部用相同十道留出題、69個真值目標、最多24個新token與greedy。表中的「回答NLL」是negative log-likelihood，也就是把模型給標準答案的機率取負自然對數：猜得越有把握且正確，代價越小。這裡在每個標準回答前文上計算下一個真值token的代價，再把全部69個有效回答／EOS位置的代價加總、除以69；不是先平均十題再平均。它只按真值評分，沒有加入教師KL，因此CE與CE+KL學生可以用同一個分母比較；不能把這列拿去和它們各自的混合訓練loss直接排名。

| 版本 | 參數數量 | 完整答案匹配／10題 | 回答NLL |
| --- | ---: | ---: | ---: |
| 寬64、兩層教師 | 141,568 | 5 | 0.5058 |
| 寬16 CE | 13,744 | 1 | 1.1134 |
| 寬16教師硬目標 | 13,744 | 1 | 1.1134 |
| 寬16 CE+KL | 13,744 | 1 | 1.2407 |
| 寬32 CE | 33,632 | 4 | 0.4393 |
| 寬32教師硬目標 | 33,632 | 4 | 0.4393 |
| 寬32 CE+KL | 33,632 | 4 | 0.5037 |

本輪同寬度蒸餾沒有提高答對數；寬32的KL學生測試代價也比CE高約0.0644。兩者都正常EOS10/10，完成匹配同為4/10，但並非同四題：CE對紅方形的`color?`答`red`、`joint?`答`square,low`；KL對這兩題答`blow`與`square,high`。與教師整串ID一致率從CE的3/10增到KL的4/10，也沒有變成更高的獨立真值正確率。

寬32的CE與KL學生更新各花約2.859與3.345秒，白盒教師快取另花0.049秒；硬回答生成另花0.827秒。本輪硬回答恰好全正確、長度也相同，所以讀到的目標數匹配。一般教師文字長短不同，固定題目索引、batch與更新次數仍可能改變有效token總數；工具逐支記錄它，不假裝總時間也匹配。這是單一seed、小型合成任務，不能由一次持平或退步推論蒸餾對所有學生無效。

域外診斷另掃GSM8K包的200道原題，要求完整聊天提示加24個生成token能放入128-token上下文，且標準答案連EOS也放得下；只找到零基索引14與94兩題，其餘198題不計分，沒有截短後配完整答案。Joy每20分鐘讀8頁，讀120頁應是5小時；John每日寫20頁，三本各400頁應用60天。教師分別答`squarene,highire,ee,low`、`square,low`，不是數字答案；教師與所有學生皆0/2、EOS2/2。這兩道完整輸入足以顯示本輪教師沒有完成這些題，不能冒稱完整GSM8K benchmark。選題記錄與逐題輸出保存在`unseen-gsm8k-complete-prompts.json`及[實報的`out_of_domain_gsm8k`](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/distillation.json)，不合預算的題目沒有被算成教師答錯。

</details>

<details>
<summary>選讀：用Python重算人工表的正確率與總時間</summary>


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
runs = [
    {
        "method": "SFT",
        "student_params": 6104,
        "steps": 100,
        "tokens": 1000,
        "student_seconds": 10,
        "teacher_seconds": 0,
        "correct": 3,
        "total": 4,
    },
    {
        "method": "distill",
        "student_params": 6104,
        "steps": 100,
        "tokens": 1000,
        "student_seconds": 10,
        "teacher_seconds": 40,
        "correct": 4,
        "total": 4,
    },
]
for run in runs:
    accuracy = run["correct"] / run["total"]
    seconds = run["student_seconds"] + run["teacher_seconds"]
    print(
        run["method"],
        "參數",
        run["student_params"],
        "步數",
        run["steps"],
        "token",
        run["tokens"],
        "正確率",
        accuracy,
        "總秒數",
        seconds,
    )

輸入字典已經有人工指定的條件與結果，程式只計算衍生指標。SFT輸出6104參數、100步、1000token、0.75、10秒；蒸餾同樣學生預算、1.0、50秒。字典中的`teacher_seconds`是教師時間，不是學生更新的一部分。

</details>
